In [2]:
import pandas as pd
import numpy as np

bond_data = pd.read_excel("Bond_Data.xlsx", sheet_name="Bond_Data")

np.random.seed(42)
rate_changes = np.random.normal(0, 0.0005, 250)

def price_bond(face_value, coupon_rate, maturity_years, ytm, frequency=1):
    coupon = face_value * (coupon_rate / 100) / frequency
    periods = int(maturity_years * frequency)
    y = (ytm / 100) / frequency
    cash_flows = np.array([coupon] * periods)
    cash_flows[-1] += face_value
    discount_factors = np.array([(1 + y) ** (-t) for t in range(1, periods + 1)])
    return np.sum(cash_flows * discount_factors)

def macaulay_duration(face_value, coupon_rate, maturity_years, ytm, frequency=1):
    coupon = face_value * (coupon_rate / 100) / frequency
    periods = int(maturity_years * frequency)
    y = (ytm / 100) / frequency
    cash_flows = np.array([coupon] * periods)
    cash_flows[-1] += face_value
    times = np.array([t / frequency for t in range(1, periods + 1)])
    pv_cash_flows = np.array([cf / ((1 + y) ** t) for t, cf in zip(range(1, periods + 1), cash_flows)])
    price = np.sum(pv_cash_flows)
    return np.sum(times * pv_cash_flows) / price

def modified_duration(face_value, coupon_rate, maturity_years, ytm, frequency=1):
    mac_dur = macaulay_duration(face_value, coupon_rate, maturity_years, ytm, frequency)
    y = (ytm / 100) / frequency
    return mac_dur / (1 + y)

def dv01_per_100(face_value, coupon_rate, maturity_years, ytm, frequency=1):
    base_price = price_bond(face_value, coupon_rate, maturity_years, ytm, frequency)
    bumped_price = price_bond(face_value, coupon_rate, maturity_years, ytm + 0.01, frequency)
    return base_price - bumped_price

def stress_test(price, modified_duration, shock_bps):
    shock_decimal = shock_bps / 10000
    price_change = -modified_duration * shock_decimal * price
    return price_change

def calculate_pnl(price, modified_duration, rate_changes):
    return -modified_duration * price * rate_changes

def calculate_var(pnl, confidence=0.95):
    return np.percentile(pnl, (1 - confidence) * 100)

scenario_shocks = {
    "Bond A": {"P100": 100, "P200": 200, "Steep": 50, "Flat": 150},
    "Bond B": {"P100": 100, "P200": 200, "Steep": 100, "Flat": 100},
    "Bond C": {"P100": 100, "P200": 200, "Steep": 150, "Flat": 50}
}

results = []

for _, row in bond_data.iterrows():
    bond_id = row["Bond ID"]
    face = row["Face Value"]
    coupon = row["Coupon Rate"]
    maturity = row["Maturity (Years)"]
    ytm = row["YTM"]
    freq = row["Payment Frequency"]
    position_size = row["Position Size"]

    price = price_bond(face, coupon, maturity, ytm, freq)
    mac_dur = macaulay_duration(face, coupon, maturity, ytm, freq)
    mod_dur = modified_duration(face, coupon, maturity, ytm, freq)
    dv01_unit = dv01_per_100(face, coupon, maturity, ytm, freq)

    units = position_size / face
    market_value = price * units
    total_dv01 = dv01_unit * units

    shock = scenario_shocks.get(bond_id, {"P100": 100, "P200": 200, "Steep": 100, "Flat": 100})

    p100_unit = stress_test(price, mod_dur, shock["P100"])
    p200_unit = stress_test(price, mod_dur, shock["P200"])
    steep_unit = stress_test(price, mod_dur, shock["Steep"])
    flat_unit = stress_test(price, mod_dur, shock["Flat"])

    p100_total = p100_unit * units
    p200_total = p200_unit * units
    steep_total = steep_unit * units
    flat_total = flat_unit * units

    pnl_unit = calculate_pnl(price, mod_dur, rate_changes)
    var_unit = calculate_var(pnl_unit, 0.95)
    var_total = var_unit * units

    results.append({
        "Bond ID": bond_id,
        "Position Size": round(position_size, 2),
        "Market Value": round(market_value, 2),
        "Price": round(price, 4),
        "Macaulay Duration": round(mac_dur, 4),
        "Modified Duration": round(mod_dur, 4),
        "DV01 per 100 Face": round(dv01_unit, 6),
        "Total DV01": round(total_dv01, 2),
        "Parallel +100bp Loss": round(p100_total, 2),
        "Parallel +200bp Loss": round(p200_total, 2),
        "Steepening Loss": round(steep_total, 2),
        "Flattening Loss": round(flat_total, 2),
        "VaR 95%": round(var_total, 2)
    })

final_df = pd.DataFrame(results)

portfolio_summary = pd.DataFrame([{
    "Bond ID": "Portfolio Total",
    "Position Size": round(final_df["Position Size"].sum(), 2),
    "Market Value": round(final_df["Market Value"].sum(), 2),
    "Price": "",
    "Macaulay Duration": "",
    "Modified Duration": "",
    "DV01 per 100 Face": "",
    "Total DV01": round(final_df["Total DV01"].sum(), 2),
    "Parallel +100bp Loss": round(final_df["Parallel +100bp Loss"].sum(), 2),
    "Parallel +200bp Loss": round(final_df["Parallel +200bp Loss"].sum(), 2),
    "Steepening Loss": round(final_df["Steepening Loss"].sum(), 2),
    "Flattening Loss": round(final_df["Flattening Loss"].sum(), 2),
    "VaR 95%": round(final_df["VaR 95%"].sum(), 2)
}])

output_df = pd.concat([final_df, portfolio_summary], ignore_index=True)

print(output_df)
output_df.to_excel("Risk_Output.xlsx", index=False)

            Bond ID  Position Size  Market Value    Price Macaulay Duration  \
0            Bond A         800000     795367.43  99.4209            1.9794   
1            Bond B        1200000    1189777.72  99.1481            2.9271   
2            Bond C        1500000    1486338.99  99.0893            4.7156   
3            Bond D        1800000    1766981.46  98.1656            6.3746   
4            Bond E        2200000    2145931.11  97.5423             8.587   
5            Bond F         600000     595353.88  99.2256            1.9823   
6            Bond G        1300000    1285503.32  98.8849             3.844   
7            Bond H        1700000    1672734.89  98.3962            5.5625   
8            Bond I        2100000    2057408.01  97.9718            7.0807   
9            Bond J        2500000    2453614.50  98.1446            9.7358   
10  Portfolio Total       15700000   15449011.31                              

   Modified Duration DV01 per 100 Face  Total DV01 